#### ENRICH `ORDERS` FROM BRONZE TO SILVER

In [0]:
orders_df = spark.table('ORDERS_SALES.BRONZE.ORDERS')
orders_df.limit(1).display()
print(orders_df.count())

In [0]:
from pyspark.sql.functions import *

# Select and cast columns properly
orders_date_df = (
    orders_df
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date",
        date_format(col("order_purchase_timestamp"), 'yyyy-MM-dd').alias("purchase_date"),
        date_format(col("order_approved_at"), 'yyyy-MM-dd').alias("approved_date"),
        date_format(col("order_delivered_carrier_date"), 'yyyy-MM-dd').alias("delivered_date"),
        date_format(col("order_estimated_delivery_date"), 'yyyy-MM-dd').alias("estimated_date"),
        "file_name",
        "file_path"

)
)

# Show one row for verification
orders_date_df.limit(1).display()

In [0]:
orders_date_trans_df = (orders_date_df
                        .select("order_id",
                                "customer_id",
                                initcap(col("order_status")).alias("order_status"),
                                "order_purchase_timestamp",
                                "order_approved_at",
                                "order_delivered_carrier_date",
                                "order_delivered_customer_date",
                                "order_estimated_delivery_date",
                                "purchase_date",
                                "approved_date",
                                "delivered_date",
                                hour(col("order_purchase_timestamp")).alias("purchase_hour"),
                                dayofweek(col("order_purchase_timestamp")).alias("purchase_weekday"),
                                date_format(col("order_purchase_timestamp"),"E").alias("purchase_weekday_name"),
                                when(col("order_delivered_customer_date").isNotNull(),hour(col("order_delivered_customer_date"))
                                    ).alias("delivery_hour"),
                                when(col("order_delivered_customer_date").isNotNull(),dayofweek(col("order_delivered_customer_date"))
                                    ).alias("delivered_weekday"),
                                when(col("order_delivered_customer_date").isNotNull(),date_format(col("order_delivered_customer_date"),"E")
                                    ).alias("delivered_weekday_name"), 
                                "file_path",
                                current_timestamp().alias("load_timestamp")
)
)

orders_date_trans_df.limit(1).display()

In [0]:
orders_date_trans_metrices_df = (orders_date_trans_df
                                 .select("*",
                                         datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")).alias("delivery_days"),
                                         when(col("order_delivered_customer_date").isNotNull(),datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date"))
                                              ).alias("delay_days")
                                         )
                                 
                                 )
orders_date_trans_metrices_df.limit(1).display()

In [0]:
orders_date_trans_purchase_metrices_df = orders_date_trans_metrices_df.select(
    "*",
    when(col("purchase_hour").between(6, 11), 'MORNING')
    .when(col("purchase_hour").between(12, 17), 'AFTERNOON')
    .when(col("purchase_hour").between(18, 22), 'EVENING')
    .otherwise('NIGHT')
    .alias("purchase_time_of_day")
)
orders_date_trans_purchase_metrices_df.limit(1).display()

In [0]:
orders_date_trans_order_status_metrices_df = (orders_date_trans_purchase_metrices_df
                                              .select("*",
                                              when(col("order_status").isin('Delivered', 'Invoice'), 'COMPLETED')
                                              .when(col("order_status").isin('Cancelled', 'Unvailable'), 'CANCELLED')
                                              .otherwise('PENDING').alias("order_status_group")
)
                                              )
orders_date_trans_order_status_metrices_df.limit(1).display()

In [0]:
orders_date_trans_approval_metrices_df = (
    orders_date_trans_order_status_metrices_df
        .select(
            "*",
            (
                (col("order_approved_at") < col("order_purchase_timestamp")) |
                (col("order_delivered_carrier_date") < col("order_approved_at")) |
                (col("order_delivered_customer_date") < col("order_delivered_carrier_date"))
            ).alias("is_invalid_timeflow")
        )
)
orders_date_trans_approval_metrices_df.limit(1).display()

In [0]:
products_enriched_df = (
    orders_date_trans_approval_metrices_df
    .select("order_id", "customer_id", "order_status", "order_status_group", "order_purchase_timestamp", "order_approved_at",
            "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date", "purchase_date",
            "approved_date", "delivered_date", "purchase_hour", "purchase_weekday", "purchase_weekday_name", "delivery_hour",
            "delivered_weekday", "delivered_weekday_name", "delivery_days", "delay_days", "purchase_time_of_day", "is_invalid_timeflow",
            "file_path", "load_timestamp")
    )
products_enriched_df.limit(1).display()

In [0]:
products_enriched_df.createOrReplaceTempView('enriched_products_temp_vw')

In [0]:
%sql
MERGE INTO ORDERS_SALES.SILVER.ORDERS_ENRICHED tgt
USING enriched_products_temp_vw src
ON (src.order_id = tgt.order_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.order_id = src.order_id,
    tgt.customer_id = src.customer_id,
    tgt.order_status = src.order_status,
    tgt.order_status_group = src.order_status_group
WHEN NOT MATCHED THEN
INSERT (order_id, 
    customer_id, 
    order_status, 
    order_status_group, 
    order_purchase_timestamp, 
    order_approved_at, 
    order_delivered_carrier_date,
    order_delivered_customer_date, 
    order_estimated_delivery_date, 
    purchase_date, 
    approved_date, 
    delivered_date, 
    purchase_hour, 
    purchase_weekday, 
    purchase_weekday_name, 
    delivery_hour, 
    delivered_weekday, 
    delivered_weekday_name, 
    delivery_days, 
    delay_days, 
    purchase_time_of_day, 
    is_invalid_timeflow, 
    origin_file_path, 
    load_timestamp
)
VALUES (src.order_id, 
src.customer_id, 
src.order_status, 
src.order_status_group, 
src.order_purchase_timestamp, 
src.order_approved_at, 
src.order_delivered_carrier_date, 
src.order_delivered_customer_date, 
src.order_estimated_delivery_date, 
src.purchase_date, 
src.approved_date, 
src.delivered_date, 
src.purchase_hour, 
src.purchase_weekday, 
src.purchase_weekday_name, 
src.delivery_hour, 
src.delivered_weekday, 
src.delivered_weekday_name, 
src.delivery_days, 
src.delay_days, 
src.purchase_time_of_day, 
src.is_invalid_timeflow, 
src.file_path, current_timestamp)

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.ORDERS_ENRICHED`

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.ORDERS_ENRICHED')